# 1: Explain the use of device cuda or cpu in Pytorch

In [1]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using {device} device")

Using cuda device


In [32]:
from torchvision.models import resnet50, ResNet50_Weights
model =resnet50(weights=ResNet50_Weights.IMAGENET1K_V2) 
model = model.to(device)
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [33]:
data = torch.randn((4,1))
print(f"Data before: {data.device}")

data = data.to(device)
print(f"Data after: {data.device}")

Data before: cpu
Data after: cuda:0


# 2: How do we prepare our data for training with using DataLoaders

https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html


In [34]:
from torch.utils.data import Dataset
from PIL import Image

# Custom Dataset Class
class ImageDataset(Dataset):
    def __init__(self, image_paths, image_labels, transform=None):
        self.image_paths = image_paths
        self.image_labels = image_labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        #Load image
        image = Image.open(self.image_paths[idx]).convert('RGB')

        # Apply transformations if provided

        if self.transform:
            image = self.transform(image)

        # Get label
        label = self.image_labels[idx]
        return image, label

In [3]:
import os

# Define dataset path for the train and the test
dataset_path_train = "santa/train"
dataset_path_test = "santa/test"

# Define the labels
labels = ["santa", "not-a-santa"]

train_path = []
train_labels = []
test_path= []
test_labels = []

# For training
for label in labels:
    folder_path = os.path.join(dataset_path_train, label)
    
    # Get all image files in the folder
    for img_file in os.listdir(folder_path):
        train_path.append(os.path.join(folder_path, img_file))
        train_labels.append(label)

# For test
for label in labels:
    folder_path = os.path.join(dataset_path_test, label)
    
    # Get all image files in the folder
    for img_file in os.listdir(folder_path):
        test_path.append(os.path.join(folder_path, img_file))
        test_labels.append(label)

In [4]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Define transformations
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])
# Create the datasets
train_dataset = ImageDataset(train_path, train_labels, transform=transform)
test_dataset = ImageDataset(test_path, test_labels, transform=transform)

# Hyperparameters
batch_size = 32
num_workers = 0

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)

print(train_loader)
print(test_loader)

NameError: name 'ImageDataset' is not defined

# 3: How can we verify that the batch images and labels are correct?

In [ ]:
import matplotlib.pyplot as plt
import torchvision
import numpy as np

## imshow method that displays images
def imshow(img, mean, std):
    # Unnormalize
    img = img * std[:, None, None] + mean[:, None, None]
    npimg = img.numpy()
    plt.figure(figsize=(8, 8))
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.axis('off')
    plt.show()
    
# Get a batch of training data
dataiter = iter(train_loader)
images, labels = next(dataiter)

# Define mean and std for unnormalization
mean = torch.tensor([0.5,0.5,0.5])
std = torch.tensor([0.5,0.5,0.5])

# Loop to display 4 images at a time
batch_size = 4

for i in range(0,len(images), batch_size):
    # Select a batch of 4 images and labels
    image_batch = images[i:i+batch_size]
    label_batch = labels[i:i+batch_size]

    # Show the images in the batch
    imshow(torchvision.utils.make_grid(image_batch), mean, std)

    # Print the labels for the current batch of images
    print('Labels: ', ', '.join(f'{label_batch[j]}' for j in range(len(label_batch))))